In [ ]:
!pip install --upgrade  setuptools 

In [1]:
import gurobipy as gp
from gurobipy import GRB
from utils.data import *

In [101]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2023, month=1, day=1)
all_stocks = ["NVDA", "AAPL", "BA", 'DB']

n = len(all_stocks)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)
#Sigma_XY = Sigma[]

r = 0.05
T = 90
S = [yf.Ticker(s).history(period="1d") for s in all_stocks]

In [102]:
Sigma

array([[0.11700434, 0.03873312, 0.01730291, 0.03181702],
       [0.03873312, 0.04632233, 0.00519544, 0.01680782],
       [0.01730291, 0.00519544, 0.03139465, 0.02279149],
       [0.03181702, 0.01680782, 0.02279149, 0.04028686]])

In [103]:
m = gp.Model()

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
Delta = m.addMVar(n, lb=0.1, ub=.5, name="Delta")
#Phi_inv = m.addMVar(n, lb=-GRB.INFINITY, name="phi of inverse CDF")
#phi = m.addMVar(n, lb=-GRB.INFINITY, name="phi")
#K = m.addMVar(n, lb=0, name="strike")
P = m.addMVar(n, lb=-GRB.INFINITY, name="put price")
#d1 = m.addMVar(n, lb=-GRB.INFINITY, name="d1")
#d2 = m.addMVar(n, lb=-GRB.INFINITY, name="d2")
var_norm = m.addVar()

for i in range(n):
    Phi_inv_expr = 4/1.7*(Delta[i] - 0.5)
    K_expr = sigma[i]*Phi_inv_expr + S[i]
    d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
    d2_expr = d1_expr - sigma[i]*np.sqrt(T)
    cdf_d1_expr = 0.5 - (1/np.sqrt(2*np.pi))*d1_expr
    cdf_d2_expr = 0.5 - (1/np.sqrt(2*np.pi))*d2_expr
    P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr
    phi_expr = 1/np.sqrt(2*np.pi)*(1-Phi_inv_expr**2/2)


    m.addConstr(P[i] == P_expr)
    m.addConstr(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv_expr+ phi_expr) - P[i] )

t = m.addVar(lb=-GRB.INFINITY)

m.addConstr(gp.quicksum([wi for wi in w]) == 1)
m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) >= var_norm * t)

Sigma_d = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    for j in range(i+1, n):
        rho_ij = Sigma[i,j] / sigma[i]*sigma[j]
        sqrt_1mrho2 = np.sqrt(1 - rho_ij**2)

        eta_i = -4/1.7*(Delta[i] - 0.5)   
        eta_j = -4/1.7*(Delta[j] - 0.5)

        phi_i = 1/np.sqrt(2*np.pi)*(1 - eta_i**2/2)
        phi_j = 1/np.sqrt(2*np.pi)*(1 - eta_j**2/2)

        Phi_eta_i = 1 - Delta[i]
        Phi_eta_j = 1 - Delta[j]

        w_ij = (eta_i - rho_ij*eta_j) / sqrt_1mrho2
        w_ji = (eta_j - rho_ij*eta_i) / sqrt_1mrho2

        Phi_w_ij = 0.5 + 1/np.sqrt(2*np.pi)*w_ij
        Phi_w_ji = 0.5 + 1/np.sqrt(2*np.pi)*w_ji

        Phi2 = Phi_eta_i*Phi_eta_j + rho_ij*phi_i*phi_j

        phi2 = phi_i*phi_j / sqrt_1mrho2

        a_i = 4/1.7*(Delta[i] - 0.5)
        a_j = 4/1.7*(Delta[j] - 0.5)

       # lhs = m.addVar(lb=-GRB.INFINITY, name=f"EZZ_{i}_{j}_lhs")
        covar_ij = m.addVar(lb=-GRB.INFINITY, name=f"covar_{i}_{j}")

        #m.addConstr(lhs == EZZ_ij * Phi2)
        print(rho_ij)
        m.addConstr(
            covar_ij == mu[i]*mu[j]+rho_ij
            + (rho_ij * a_i * phi_i * Phi_w_ji
            + rho_ij * a_j * phi_j * Phi_w_ij
            + (1 - rho_ij**2) * phi2)/Phi2 -  mu_d[i]*mu_d[j]
        )

        Sigma_d[i, j] = covar_ij      
        Sigma_d[j, i] = covar_ij

    Phi_inv_i = 4/1.7 * (Delta[i] - 0.5)                          
    phi_i = 1/np.sqrt(2*np.pi) * (1 - Phi_inv_i**2 / 2)       
    
    var_i = m.addVar(lb=-GRB.INFINITY, name=f"var_{i}")

    m.addConstr(
        var_i == sigma[i]**2 *                                  \
        (Delta[i] * Phi_inv_i**2 *(1-Delta[i])                  \
        + phi_i*(-2*Delta[i]*Phi_inv_i + Phi_inv_i - phi_i)     \
        - Delta[i] + 1)
    )

    Sigma_d[i, i] = var_i


m.addConstr(var_norm == gp.quicksum([Sigma_d[i, i] * w[i]**2 for i in range(n)]) + \
                        2*gp.quicksum([Sigma_d[i, j] * w[i]*w[j] for i in range(n) for j in range(n) if j > i]))

m.setObjective(t, GRB.MAXIMIZE)

m.optimize()


#m.addConstrs(Phi_inv[i] == -1/1.7*nlfunc.log(1/Delta[i]-1) for i in range(n))
"""
m.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))

m.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))
m.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))

m.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))
m.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))

m.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))
m.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))

m.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))

"""


0.024371187483918137
0.00896284439655937
0.018669821719700292
0.004277153079733072
0.015674654773622655
0.02581824361298533
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.0.0 24A348)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 28 columns and 4 nonzeros (Max)
Model fingerprint: 0x18e8de29
Model has 1 linear objective coefficients
Model has 9 quadratic constraints
Model has 11 general nonlinear constraints (138 nonlinear terms)
Variable types: 28 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [2e-06, 1e+00]
  QLMatrix range   [1e-02, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e+00, 1e+00]
  QRHS range       [2e-02, 2e+02]
  NLCon coe range  [5e-04, 6e+00]

Added 152 variables to disaggregate expressions
Presolve time: 0.00s
Presolved: 483 rows, 208 columns, 1274 non

'\nm.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))\n\nm.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))\nm.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))\n\nm.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))\nm.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))\n\nm.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))\nm.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))\n\nm.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))\n\n'

In [104]:
weights = np.zeros(n)

var = m.getVars()

weights = [v.X for v in var if "weight" in v.VarName]

deltas = [v.X for v in var if "Delta" in v.VarName]
P = [v.X for v in var if "price" in v.VarName]
d = [v.X for v in var if "d1" in v.VarName]
weights, deltas, P, d

([0.0, 0.0, 0.0, 1.0],
 [0.5, 0.5, 0.5, 0.1],
 [142.5586160130936,
  214.69500522987755,
  207.65128892317182,
  26.714851803601547],
 [])

In [107]:
vars = m.getVars()

var = np.array([v for v in vars if  v.VarName[:4] == "var_"])
covar = np.array([v for v in vars if  v.VarName[:6] == "covar_"])

Sigma_result = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    for j in range(i+1, n):
        print([v for v in vars if v.VarName == f"covar_{i}_{j}"])
        Sigma_result[i, j] = np.array([v for v in vars if v.VarName == f"covar_{i}_{j}"])[0]
Sigma_result

[<gurobi.Var covar_0_1 (value -30530.71349939141)>]
[<gurobi.Var covar_0_2 (value -29536.556458843672)>]
[<gurobi.Var covar_0_3 (value -3798.4367839122115)>]
[<gurobi.Var covar_1_2 (value -44525.40908754357)>]
[<gurobi.Var covar_1_3 (value -5726.087574911979)>]
[<gurobi.Var covar_2_3 (value -5539.618828530729)>]


array([[0, <gurobi.Var covar_0_1 (value -30530.71349939141)>,
        <gurobi.Var covar_0_2 (value -29536.556458843672)>,
        <gurobi.Var covar_0_3 (value -3798.4367839122115)>],
       [0, 0, <gurobi.Var covar_1_2 (value -44525.40908754357)>,
        <gurobi.Var covar_1_3 (value -5726.087574911979)>],
       [0, 0, 0, <gurobi.Var covar_2_3 (value -5539.618828530729)>],
       [0, 0, 0, 0]], dtype=object)

In [108]:
import plotly.express as px

px.line(benchmark(all_stocks, weights, "SPY", end, dt.datetime.today()))

In [ ]:
px.line(benchmark(all_stocks, np.ones(len(all_stocks))/len(all_stocks), "SPY", end, dt.datetime.today()))

In [ ]:
sigma = [np.std(get_hist(s, start, end).Open.to_numpy()) for s in all_stocks]

sigma

In [ ]:
import plotly.express as px
import sympy as sp

K = sp.Symbol("K")
S = S[0]
sigma = sigma[0]

d1 = (1-K/S+T*(r+sigma**2/2))/(sigma*np.sqrt(T)) 

sp.plot(d1)

In [ ]:
#if m.status == GRB.INFEASIBLE:
m.computeIIS()
m.write("model.ilp")
for c in m.getConstrs():
    if c.IISConstr:
        print(f"Infeasible constraint: {c.ConstrName}")
for v in m.getVars():
    if v.IISLB:
        print(f"Infeasible lower bound: {v.VarName} lb={v.LB}")
    if v.IISUB:
        print(f"Infeasible upper bound: {v.VarName} ub={v.UB}")
    
